# Programmazione a oggetti in OCaml (cenni)


OCaml supporta anche la programmazione a oggetti

* Un **oggetto** in OCaml è un *valore*, costituito da campi e metodi che rappresentano rispettivamente lo stato e il comportamento dell'oggetto

* Sebbene esistano costrutti linguistici per la definizione di **classi**, gli oggetti possono essere creati direttamente, senza prima specificare una classe (come in JavaScript)

* Il tipo di un oggetto è determinato esclusivamente dai metodi che esso contiene ed espone (i campi non influiscono sul tipo)


Vediamo un esempio di oggetto che realizza una pila, in cui `v` rappresenta lo stato interno (modificabile) dello stack

In [35]:
(* oggetto che realizza uno stack *)
let s = object

    (* campo mutabile che contiene la rappresentazione dello stack *)
    val mutable v = [0; 2]  (* Assumiamo per ora inizializzato non vuoto *)
        
    (* metodo pop *)
    method pop =
        match v with
        | hd :: tl ->
            v <- tl;
            Some hd
        | [] -> None
        
    (* metodo push *)
    method push hd =
        v <- hd :: v
end ;;


val s : < pop : int option; push : int -> unit > = <obj>


* Il tipo inferito è formato da object-type e i solo metodi: è un oggetto che possiede un metodo `pop` che restituisce `int option`, e un metodo `push` che accetta un `int` e restituisce `unit`

* I nomi dei metodi appaiono in ordine alfabetico per rendere efficienti i confronti


**NOTA SINTATTICA 1**:
Nei metodi senza parametri non è necessario aggiungere `()`

**NOTA SINTATTICA 2**: I campi dell’oggetto sono visibili nei metodi (non serve `this`)

Dopo aver definito l'oggetto `s`, è possibile interagire con esso chiamando i metodi *pubblici* `push` e `pop`

Notare che `v` è invece un campo privato (*information hiding*), accessibile solo all’interno dell’oggetto

In [4]:
s#pop;;  (* Some 0 *)
s#pop;;  (* Some 2 *)
s#pop;;  (* None *)
s#push 9;;
s#pop;;  (* Some 9 *)
s#v;;

- : int option = Some 0


- : int option = Some 2


- : int option = None


- : unit = ()


- : int option = Some 9


error: compile_error

L'invocazione di metodo si fa con la #-notation invece che con la dot-notation
(no overloading)

### Piccola digressione: type weakening

Che cosa succede se nell’oggetto `s` inizializziamo `v` come lista vuota?
Che tipo viene inferito per l’oggetto?


In [9]:
let s = object

   val mutable v = []  (* lista vuota! *)

   method pop =
        match v with
        | hd :: tl ->
            v <- tl;
            Some hd
        | [] -> None
        
    (* metodo push *)
    method push hd =
        v <- hd :: v
end ;;

val s : < pop : '_weak1 option; push : '_weak1 -> unit > = <obj>


Il tipo inferito contiene variabili di tipo, ma non è veramente polimorfo:
lo è solo temporaneamente

* Benché sia mutabile, la variabile `v` non potrà avere tipi diversi in momenti diversi dell’esecuzione
* L’oggetto `s` dovrebbe avere tipo
		 < pop : t option; push : t -> unit > 
  per un qualche tipo concreto `t`
* `t` non è però noto al momento della dichiarazione della variabile mutabile, quindi il type checker *indebolisce* temporaneamente il tipo inferito includendo delle variabili di tipo

* Appena possible (al primo utilizzo) il tipo di `s` sarà ricalcolato andando a istanziare *definitivamente* la variabile provvisoria con un tipo concreto

In [10]:
s;;
s#pop ;;
s#push 5 ;;
s ;;
s#pop ;;
s#push "ciao" ;;

- : < pop : '_weak1 option; push : '_weak1 -> unit > = <obj>


- : '_weak1 option = None


- : unit = ()


- : < pop : int option; push : int -> unit > = <obj>


- : int option = Some 5


error: compile_error

`push` fornisce informazioni che consentono di istanziare `‘_weak`

## Costruzione di oggetti tramite funzioni

* Gli oggetti possono essere costruiti tramite funzioni

* Vediamo una funzione `stack` che costruisce oggetti inizializzati con un parametro `init`, il cui valore deve essere una lista (cioè `init : 'a list`)


In [37]:
(* funzione "stack" che costruisce oggetti inizializzati con parametro init *)
let stack init = object
    val mutable v = init (* valore iniziale *)

    method pop =
        match v with
        | hd :: tl ->
            v <- tl;
            Some hd
        | [] -> None
        
    method push hd =
        v <- hd :: v
end ;; 

val stack : 'a list -> < pop : 'a option; push : 'a -> unit > = <fun>


La funzione `stack` è (veramente) 
polimorfa!

In [39]:
let s = stack [3; 2; 1] ;; (* Crea uno stack inizializzato con [3; 2; 1] *)
s#pop ;;                   (* Rimuove e restituisce 3 *)
let s = stack [] ;;  (* con [] ancora type weakening... *)
(* ... ma basta una type annotation del parametro attuale per forzare l’instanziazione al tipo che preferiamo *)
let s = stack ([]: int list) ;; 

val s : < pop : int option; push : int -> unit > = <obj>


- : int option = Some 3


val s : < pop : '_weak4 option; push : '_weak4 -> unit > = <obj>


val s : < pop : int option; push : int -> unit > = <obj>


## Polimorfismo di oggetti

Quando si definisce una funzione che prende un oggetto come parametro, il tipo dell’oggetto viene inferito dai metodi che la funzione chiama su di esso

Questo accade indipendentemente dal fatto che l’oggetto sia già stato definito o meno!


In [12]:
let area sq = sq#width * sq#width ;;

let minimize sq = sq#resize 1 ;; (* resize non è definito *)

let limit sq = if (area sq) > 100 then minimize sq ;;

val area : < width : int; .. > -> int = <fun>


val minimize : < resize : int -> 'a; .. > -> 'a = <fun>


val limit : < resize : int -> unit; width : int; .. > -> unit = <fun>


La notazione `< width : int` indica che l’oggetto atteso, da passare ad area come parametro, deve contenere almeno il metodo `width`, ed *eventualmente* anche altro (espresso tramite i puntini   `..`  )


Vedendo `sq#resize 1`, si deduce che

* `sq` deve avere un metodo `resize` applicabile a un `int`; quindi `resize : int -> 'a`
* `minimize` accetta qualunque oggetto che abbia almeno un metodo `resize` applicabile a un `int` 

Dopo aver visto la definizione di `limit sq`, 

* Dato che `minimize` e quindi `resizize` è usata dentro un `if` che non restituisce valore, allora si ha
`resize:  int -> unit` 
* Dato che `limit` usa `area`, allora deve avere anche un metodo `width`

**Flessibilità**: la funzione non richiede che l’oggetto sia stato definito in anticipo con uno specifico tipo:
basta che l’oggetto passato alla funzione contenga i metodi richiesti, senza vincoli aggiuntivi

Una funzione può quindi operare su oggetti di tipi diversi, purché questi soddisfino i requisiti di metodo

Il seguente oggetto «quadrato» può essere passato a tutte le funzioni viste, poiché soddisfa i requisiti di tipo richiesti da ciascuna di queste funzioni. Infatti implementa i metodi `width` and `resize`


In [60]:
let quadrato = object
  val w = ref 30
  method width = !w
  method color = "red"
  method resize n = w := n
end ;;

val quadrato : < color : string; resize : int -> unit; width : int > = <obj>


### Structural subtyping

Si applica la regola di *subsumption* 
$$
\frac{\Gamma \vdash e : S \qquad S <: T}{\Gamma \vdash e : T}
$$


che consente di tipare un'espressione con un suo supertipo: se un’espressione $e$ ha tipo $S$, e $S$ è sottotipo di $T$, allora $e$ può essere considerata di tipo $T$


Ad esempio, abbiamo

$$
\langle \texttt{color : string;} \ 
         \texttt{resize : int → unit;} \
         \texttt{width : int} \rangle
\quad <: \quad
\langle \texttt{width : int} \rangle
$$

Rispetto al secondo tipo, il primo rappresenta un oggetto più *più specifico*, cioè più ricco di proprietà e metodi 
 
In termini di valori, l’insieme descritto dal primo tipo è un sottoinsieme di quello descritto dal secondo:

* tra tutti gli oggetti che hanno almeno un metodo `< width : int >` ci sono anche oggetti di tipo `< color : string; resize : int -> unit; width : int >` (ne sono un sottoinsieme)

Pertanto possiamo dire il primo è *sottotipo* del secondo

Ad esempio, abbiamo

$$
\langle \texttt{color : string;} \ 
         \texttt{resize : int → unit;} \
         \texttt{width : int} \rangle
\quad <: \quad
\langle \texttt{width : int} \rangle
$$

Rispetto al secondo tipo, il primo rappresenta un oggetto più *più specifico*, cioè più ricco di proprietà e metodi 
 
In termini di valori, l’insieme descritto dal primo tipo è un sottoinsieme di quello descritto dal secondo:

* tra tutti gli oggetti che hanno almeno un metodo `< width : int >` ci sono anche oggetti di tipo `< color : string; resize : int -> unit; width : int >` (ne sono un sottoinsieme)

Pertanto possiamo dire il primo è *sottotipo* del secondo

Grazie alla regola di *subsumption*, possiamo quindi derivare:

$$
\Gamma \vdash \texttt{quadrato} : 
\langle \texttt{width : int} \rangle
$$

e concludere che l’oggetto `quadrato` può essere passato come argomento 
a una funzione che si aspetta un parametro di quel tipo, ad esempio la funzione `area` 

Lo stesso ragionamento vale per funzioni come `minimize` e `limit`

Vediamo sull'esempio:

In [61]:
area quadrato ;; (* calcola l'area del quadrato *)
limit quadrato ;; (* controlla le dimensioni e minimizza se serve *)
area quadrato ;; (* calcola l'area del quadrato *)

- : int = 900


- : unit = ()


- : int = 1


La notazione con i puntini
`< width : int, .. >`
usata dall’interprete OCaml nell’inferire il tipo del parametro formale di area enfatizza lo structural subtyping

* La funzione accetta un qualunque sottotipo di `< width : int >`, ossia qualunque oggetto che contenga almeno il metodo `width`

* Rappresenta una nuova forma di *polimorfismo (sugli oggetti)* che consente di trattare diversi oggetti con la stessa funzione, purché abbiano i metodi richiesti, indipendentemente dagli altri metodi presenti

* I puntini `..` possono essere considerati come una *variabile di tipo* special (*row variable*), che si può istanziare con una lista di altri metodi potenzialmente da aggiungere a `width`

Riassumendo:

* la regola di subsumption consente di “salire” nella gerarchia dei tipi: se un oggetto ha un tipo più specifico, può essere trattato come un’istanza di un tipo più generale

* questo è il cuore del *subtyping strutturale*: un oggetto con più campi può essere usato dove serve un oggetto con meno campi (purché compatibili)

* in pratica: un oggetto con più capacità può sostituirne uno con meno, ma non viceversa


## Coercion di tipi oggetto

Al di là del passaggio dei parametri a una funzione, ci sono numerose altre situazioni in cui il subtyping degli oggetti si rende utile

Supponiamo di definire i seguenti tipi oggetto (tramite `type`)

In [62]:
type shape = < area : float >
type square = < area : float; width : int >

type shape = < area : float >


type square = < area : float; width : int >


È chiaro che `square` sia un sottotipo di `shape` (contiene dei metodi in più), quindi è lecito pensare che ovunque si possa usare un oggetto di tipo `shape` si possa usare al suo posto un oggetto di tipo `square` (*principio di sostituzione*, ne riparleremo…)

Facciamo una prova… definiamo funzioni costruttore per i due tipi:

In [84]:
(* costruttore di oggetti di tipo shape *)
let shape (a:float): shape = object
  method area = a
end ;;

(* costruttore di oggetti di tipo square *)
let square (w:int): square = object
  method area = (float_of_int) (w * w)
  method width = w
end ;;

val shape : float -> shape = <fun>


val square : int -> square = <fun>


Proviamo ad aggiungere uno `square` a una lista di `shape`:

In [85]:
let lis1  = [shape 10.0; shape 20.0] ;;

let lis2  = square 5 :: lis1 ;;

val lis1 : shape list = [<obj>; <obj>]


error: compile_error

Non funziona: serve forzare il tipo

## Coercion di tipi oggetto: operatore `:>`

Serve una *type coerction* (*conversione di tipo*) esplicita, tramite l’operatore `:>`


La type coercion `e :> t` forza il type checker a trattare l’espressione `e` come se fosse di tipo `t`

In [65]:
let lis2  = ( square 5 :> shape ) :: lis1 ;;

val lis2 : shape list = [<obj>; <obj>; <obj>]


Dopo questa conversione,
`lis2` è una lista omogenea di oggetti `shape`

`t` deve essere un tipo più generale (un supertipo, con meno metodi) del tipo originale di `e`

* (square :> shape): upcast consentito
* (shape  :> square): downcast non consentito in OCaml (in Java è invece consentito) 

In [66]:
let (x:shape) = ((square 2) :> shape) ;;    (* OK *)

val x : shape = <obj>


In [67]:
let (y:square) = ((shape 4.0) :> square) ;; (* ERRORE!! *)

error: compile_error

Analogamente potremmo introdurre un tipo `rectangle`, sottotipo di `square` e quindi di `shape`

`rectangle <: square <: shape`

In [76]:
type rectangle = < area : float; width : int; height : int >

(* costruttore di oggetti di tipo rectangle *)
let rectangle (w : int) (h : int):rectangle = object
  method area = float_of_int (w * h)
  method width = w
  method height = h
end ;;

type rectangle = < area : float; height : int; width : int >


val rectangle : int -> int -> rectangle = <fun>


## Varianza in OCaml

**Attenzione**
Il sottotipaggio sugli oggetti non si estende *automaticamente* ai contenitori, come le liste o gli array

**Liste**

Anche se `square` è sottotipo di `shape`, una `square list` **non** accetta un elemento del tipo più generale `shape`. 

Tuttavia ricorrendo alla type coercion le liste doventano *covarianti*, ovvero

* se `A <: B`       vale      `A list  <:  B list`

Vediamo un esempio

In [101]:
(* lista di quadrati *)
let squares = [square 2; square 3];;
(* squares : square list *)
let mixed = (shape 10.0) :: squares;;

val squares : square list = [<obj>; <obj>]


error: compile_error

Se una `square list` fosse anche una `shape list`
potremmo provare a inserire una `shape` in una lista di `square`

`let mixed = (shape 10.0) :: squares;;`

ma questo porterebbe a un **errore**: il primo elemento sarebbe infatti di tipo `shape` e non avrebbe `width`

Con la conversione esplicita invece funziona e le liste diventano covarianti:
una `square list` può essere coerentemente trattata come `shape list` tramite coercion esplicita (`:>`).

In [100]:
(* squares : square list *)
let shapes: shape list = (squares :> shape list);;
let mixed = (shape 10.0) :: shapes;;

val shapes : shape list = [<obj>; <obj>]


val mixed : shape list = [<obj>; <obj>; <obj>]


**Array**

In OCaml le liste possono essere covarianti grazie all'immutabilità delle liste, mentre gli array, essendo mutabili (puoi sostituire un elemento in qualsiasi posizione), sono invarianti. 

Non sarebbe sicuro trattare uno `square array` come uno `shape array`, perché consentirebbe di memorizzare `shape` non quadrate in quello che dovrebbe essere uno `square array`. 

In quel caso, l’array, che dovrebbe contenere solo `square`, potrebbe avere elementi che non hanno il metodo `width`

Analogamente, se inserissi un elemento `rectangle` invece l'elemento perderebbe `height`, poiché l’array dovrebbe essere considerato composto solo da `square`

OCaml lo riconosce e non consente la coercion:

In [102]:
let square_array: square array = [| square 10; square 20 |];;
let shape_array: shape array = (square_array :> shape array);;

val square_array : square array = [|<obj>; <obj>|]


error: compile_error

In [98]:
let square_array: square array = [| square 10; square 20 |];;
let shape_array: shape array = (square_array :> rectangle array);;

val square_array : square array = [|<obj>; <obj>|]


error: compile_error

**Functions**

Una funzione con tipo `square -> string` non può essere utilizzata con tipo `shape -> string` perché si aspetta che il suo argomento sia uno `square` e non saprebbe cosa fare, ad esempio con un `circle`. 

Tuttavia, una funzione con tipo `shape -> string` può essere utilizzata in modo sicuro con tipo `square -> string`.

In [103]:
let shape_to_string: shape -> string =
  fun s -> Printf.sprintf "Shape(%F)" s#area;;
let square_to_string: square -> string =
  (shape_to_string :> square -> string);;

val shape_to_string : shape -> string = <fun>


val square_to_string : square -> string = <fun>


Per maggiori dettagli vedere https://dev.realworldocaml.org/objects.html

## Polimorfismo di oggetti VS principio di sostituzione

Questo esempio mostra che i due concetti di: 
* polimorfismo sugli oggetti  (ad esempio, una funzione che prende oggetti con almeno i metodi richiesti) 
* principio di sostituzione (ad esempio, un oggetto di un tipo più specifico si può usare ovunque serva un oggetto di un tipo più generale)

sebbene tra loro collegati, vengono trattati in OCaml in due modi diversi:

* il primo è supportato direttamente grazie al subtyping strutturale, mentre 
* per il secondo è richiesta infatti la type coercion esplicita: ogni cambiamento di tipo deve essere cioè dichiarato in modo esplicito dal programmatore


Questo avviene sempre perché il type checker di OCaml, come già visto con i tipi primitivi, non effettua conversioni di tipo implicite 


In Java vedremo che sarà possibile effettuare delle coercizioni di tipo da un supertipo a un sottotipo (ad esempio, trasformare uno `shape` in uno `square`)

Questo sarà possibile grazie ai controlli dinamici di tipo eseguiti dall'interprete della JVM a run time, che risultano più semplici grazie all'uso del nominal subtyping

## Classi in OCaml

* Abbiamo visto che OCaml consente di lavorare direttamente con gli oggetti (in stile object-based)

* Tuttavia, i meccanismi di forza della programmazione OO derivano dall’ereditarietà

* Abbiamo visto in JavaScript che realizzare meccanismi di ereditarietà lavorando direttamente con gli oggetti richiede di usare tecniche tipo i prototipi, il cui funzionamento è complicato…

* Per questo OCaml introduce anche dei costrutti di **classe**: intuitivamente, una classe è la «ricetta» che descrive come creare oggetti di un certo tipo

In [29]:
class istack = object  (* classe per stack di interi *)
  val mutable v = [0; 2]  (* inizializzato non vuoto *)
  method pop =
    match v with
    | hd :: tl ->
        v <- tl;
        Some hd
    | [] -> None 
  method push hd =
    v <- hd :: v
end ;;

class istack :
  object
    val mutable v : int list
    method pop : int option
    method push : int -> unit
  end


Si istanzia con `new`

In [31]:
let s = new istack ;; (* istack alias per il tipo *)
s#pop ;;

val s : istack = <obj>


- : int option = Some 0


## Classi parametriche e polimorfe

Una classe può prevedere parametri di
* costruzione (ad esempio, `init`) da passare al momento dell’istanziazione

* tipo (ad esempio, `'a`) che la rendono polimorfa

Nell'esempio:

* `['a] stack` indica che `stack` è una classe polimorfica che funziona con liste di tipo `'a`, rendendola generica

In [47]:
class ['a] stack init = object  (* classe polimorfa per stack *)
  val mutable v : 'a list = init   (* init è parametro costruttore *)
  method pop =
    match v with
    | hd :: tl ->
        v <- tl;
        Some hd
    | [] -> None 
  method push hd =
    v <- hd :: v
end ;;

class ['a] stack :
  'a list ->
  object
    val mutable v : 'a list
    method pop : 'a option
    method push : 'a -> unit
  end


In [48]:
let s = new stack ["pippo"] ;;
s#pop ;;

val s : string stack = <obj>


- : string option = Some "pippo"


## Classi e tipi oggetto

La definizione di una classe introduce anche un tipo con lo stesso nome

Si tratta però solo di un *alias* (struttrale) del tipo-oggetto che si otterrebbe costruendo gli oggetti direttamente

In [49]:
let s = new stack ["pippo"] ;;
(* string stack è un alias per
   < pop : string option; push : string -> unit >  *)

val s : string stack = <obj>


In OCaml due oggetti provenienti da classi diverse, ma con gli stessi metodi e tipi di metodi, appartengono allo stesso tipo anche se provengono da classi diverse

## Ereditarietà 

L’ereditarietà è una funzionalità realizzata tramite opportuni costrutti linguistici che consente di definire una classe (o, più in generale, una tipologia di oggetti) sulla base di un’altra esistente

* I linguaggi class-based, consentono di definire una classe come estensione di un’altra 

La nuova classe:

 * eredita tutti i membri (valori e metodi) della precedente,

 * con la possibilità di aggiungerne altri o ridefinirne alcuni (overriding)
 
 
Anche in OCaml è prevista l’ereditarietà tra classi,
realizzata tramite opportuni costrutti linguistici dedicati
(`inherit`, `super`, ecc.)

### Esempio di aggiunta 

In [50]:
class sstack init = object  (* classe per stack di stringhe *)
 inherit [string] stack init (* eredita da stack *)

method concat = (* aggiunge un nuovo metodo *)
  List.fold_left (^) "" v
end ;;


class sstack :
  string list ->
  object
    val mutable v : string list
    method concat : string
    method pop : string option
    method push : string -> unit
  end


In [51]:
let b = new sstack [" ";"world!"] ;;
b#push "Hello" ;;
b#concat ;;

val b : sstack = <obj>


- : unit = ()


- : string = "Hello world!"


### Esempio di overriding

In [52]:
(* classe per stack di int che raddoppia i valori inseriti *)
class double_stack init = object
   (* super è l’oggetto da estendere in fase di istanziazione *)
    inherit [int] stack init as super

        method push hd =       (* ridefinisce un metodo *)
        super#push ( hd * 2 )
end ;;

class double_stack :
  int list ->
  object
    val mutable v : int list
    method pop : int option
    method push : int -> unit
  end


In [43]:
let ds = new double_stack [] ;;
ds#push 5 ;;
ds#pop ;;

val ds : double_stack = <obj>


- : unit = ()


- : int option = Some 10


## Conclusioni

OCaml combina costrutti linguistici tipicamente object-based con costrutti tipicamente class-based

* I costrutti object-based favoriscono la definizione di object-types e supportano lo structural subtyping 
* La non modificabilità della struttura degli oggetti (a differenza di JavaScript) consente tuttavia di effettuare controlli di tipo a tempo di compilazione
* I costrutti class-based rendono naturale la definizione di meccanismi di ereditarietà (e altro…), garantendo una gerarchia chiara tra le classi 

OCaml prevede anche altri costrutti di OOP, che consentono di trattare aspetti importanti quali:

* Interfacce
* Classi parzialmente definite (classi astratte)
* Iteratori
…

Non vedremo tutti questi aspetti, ma li tratteremo in Java
